In [25]:
import pandas as pd
import json
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_columns', None)

In [ ]:
df_terms = pd.read_parquet('cleaned_aws_terms.parquet')
df_products = pd.read_parquet('cleaned_aws_products.parquet')

In [ ]:
print("Terms shape:", df_terms.shape)
print("Products shape:", df_products.shape)
df_products.head(3)

Terms shape: (7234, 5)
Products shape: (9031, 25)


,sku,attributes.servicecode,attributes.transferType,attributes.fromLocation,attributes.fromLocationType,attributes.toLocation,attributes.toLocationType,attributes.usagetype,attributes.operation,attributes.fromRegionCode,attributes.servicename,attributes.toRegionCode,productFamily,attributes.location,attributes.locationType,attributes.availability,attributes.storageClass,attributes.volumeType,attributes.durability,attributes.regionCode,attributes.feeCode,attributes.feeDescription,attributes.group,attributes.groupDescription,attributes.overhead
0,UBHUUCQA2BAWFNFA,AmazonS3,IntraRegion Outbound,Asia Pacific (Osaka),AWS Region,Asia Pacific (Osaka),AWS Region,APN3-APN3-S3RTC-Out-Bytes,,ap-northeast-3,Amazon Simple Storage Service,ap-northeast-3,None,None,None,None,None,None,None,None,None,None,None,None,None
1,5PP56XDP5UX3QP8N,AmazonS3,InterRegion Inbound,Canada West (Calgary),AWS Region,Asia Pacific (Melbourne),AWS Region,APS6-CAN2-S3RTC-In-Bytes,,ca-west-1,Amazon Simple Storage Service,ap-southeast-4,None,None,None,None,None,None,None,None,None,None,None,None,None
2,7H35Q77CGKE5UBSJ,AmazonS3,None,None,None,None,None,CAN2-TimedStorage-ByteHrs,,None,Amazon Simple Storage Service,None,Storage,Canada West (Calgary),AWS Region,99.99%,General Purpose,Standard,99.999999999%,ca-west-1,None,None,None,None,None


In [28]:
if 'attributes.servicename' in df_products.columns:
    df_products.drop(columns=['attributes.servicename'], errors='ignore', inplace=True)

print (f"Dimensions (rows, cols): {df_products.shape}")

Dimensions (rows, cols): (9031, 24)


In [29]:
df_s3_storage = df_products[df_products['productFamily'] == 'Storage'].reset_index(drop=True)

df_s3_operations = df_products[df_products['productFamily'] != 'Storage'].reset_index(drop=True)

##### Ανάλυση Storage

In [30]:
id_columns_s3_stor = ['sku', 'productFamily', 
                    'attributes.servicecode', 
                    'attributes.location', 
                    'attributes.locationType']

In [31]:
feature_columns_s3_stor = ['attributes.usagetype',
                        'attributes.storageClass',
                        'attributes.volumeType',
                        'attributes.availability',
                        'attributes.durability']

In [32]:
total_s3_stor_cols = id_columns_s3_stor + feature_columns_s3_stor

remaining_cols_s3_stor = [col for col in df_s3_storage.columns if col not in total_s3_stor_cols]

df_final_s3_storage = df_s3_storage[total_s3_stor_cols].copy()
df_final_s3_storage['additional_attributes'] = df_s3_storage[remaining_cols_s3_stor].apply(
    lambda row: json.dumps({k: v for k, v in row.to_dict().items() if pd.notna(v)}), 
    axis=1
)

In [ ]:
df_master_s3_storage = pd.merge(
    df_final_s3_storage, 
    df_terms, 
    on='sku', 
    how='inner'
).reset_index(drop=True)


In [34]:
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_columns', None)
df_master_s3_storage.head()

,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.usagetype,attributes.storageClass,attributes.volumeType,attributes.availability,attributes.durability,additional_attributes,rateCode,description,unit,priceUSD
0,7H35Q77CGKE5UBSJ,Storage,AmazonS3,Canada West (Calgary),AWS Region,CAN2-TimedStorage-ByteHrs,General Purpose,Standard,99.99%,99.999999999%,"{""attributes.operation"": """", ""attributes.regio...",7H35Q77CGKE5UBSJ.JRTCKXETXF.PGHJ3S3EYE,$0.025 per GB - first 50 TB / month of storage...,GB-Mo,0.02500
1,APFBAMJG3MBZYQTF,Storage,AmazonS3,EU (Milan),AWS Region,EUS1-TimedStorage-INT-DAA-ByteHrs,Intelligent-Tiering,IntelligentTieringDeepArchiveAccess,N/A,N/A,"{""attributes.operation"": """", ""attributes.regio...",APFBAMJG3MBZYQTF.JRTCKXETXF.6YS6EN2CT7,$0.0018 per Gigabyte Month for TimedStorage-IN...,GB-Mo,0.00180
2,5GXC76ATKHRPXPN9,Storage,AmazonS3,Asia Pacific (New Zealand),AWS Region,APS8-TimedStorage-INT-IA-ByteHrs,Intelligent-Tiering,Intelligent-Tiering Infrequent Access,N/A,N/A,"{""attributes.operation"": """", ""attributes.regio...",5GXC76ATKHRPXPN9.JRTCKXETXF.6YS6EN2CT7,$0.01449 per GB-Mo of Storage in Intelligent-T...,GB-Mo,0.01449
3,QEKB34JVNC6UH7GT,Storage,AmazonS3,Asia Pacific (Malaysia),AWS Region,APS7-TimedStorage-SIA-ByteHrs,Infrequent Access,Standard - Infrequent Access,99.9%,99.999999999%,"{""attributes.operation"": """", ""attributes.regio...",QEKB34JVNC6UH7GT.JRTCKXETXF.6YS6EN2CT7,$0.01242 per GB-Month of storage used in Stand...,GB-Mo,0.01242
4,A74JTBSPN4BU5EBZ,Storage,AmazonS3,Asia Pacific (Hyderabad),AWS Region,APS5-TimedStorage-INT-AA-ByteHrs,Intelligent-Tiering,IntelligentTieringArchiveAccess,N/A,N/A,"{""attributes.operation"": """", ""attributes.regio...",A74JTBSPN4BU5EBZ.JRTCKXETXF.6YS6EN2CT7,$0.0045 per Gigabyte Month for TimedStorage-IN...,GB-Mo,0.00450


#### Aνάλυση λοιπών πεδίων storage

In [37]:
id_columns_s3_ops = ['sku', 'productFamily', 
                    'attributes.servicecode', 
                    'attributes.location', 
                    'attributes.locationType',
                    'attributes.regionCode']

In [38]:
feature_columns_s3_ops = ['attributes.usagetype', 'attributes.operation', 'attributes.transferType',
                        'attributes.fromLocation', 'attributes.fromLocationType', 'attributes.toLocation',
                        'attributes.toLocationType', 'attributes.fromRegionCode', 'attributes.toRegionCode',
                        'attributes.feeCode', 'attributes.feeDescription', 'attributes.group',
                        'attributes.groupDescription', 'attributes.storageClass','attributes.volumeType']

In [39]:
total_s3_ops_cols = id_columns_s3_ops + feature_columns_s3_ops

remaining_cols_s3_ops = [col for col in df_s3_operations.columns if col not in total_s3_ops_cols]

df_final_s3_operations = df_s3_operations[total_s3_ops_cols].copy()
df_final_s3_operations['additional_attributes'] = df_s3_operations[remaining_cols_s3_ops].apply(
    lambda row: json.dumps({k: v for k, v in row.to_dict().items() if pd.notna(v)}), 
    axis=1
)

In [ ]:
df_master_s3_operations = pd.merge(
    df_final_s3_operations, 
    df_terms, 
    on='sku', 
    how='inner'
).reset_index(drop=True)

In [41]:
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_columns', None)
df_master_s3_operations.head()

,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.regionCode,attributes.usagetype,attributes.operation,attributes.transferType,attributes.fromLocation,attributes.fromLocationType,attributes.toLocation,attributes.toLocationType,attributes.fromRegionCode,attributes.toRegionCode,attributes.feeCode,attributes.feeDescription,attributes.group,attributes.groupDescription,attributes.storageClass,attributes.volumeType,additional_attributes,rateCode,description,unit,priceUSD
0,UBHUUCQA2BAWFNFA,None,AmazonS3,None,None,None,APN3-APN3-S3RTC-Out-Bytes,,IntraRegion Outbound,Asia Pacific (Osaka),AWS Region,Asia Pacific (Osaka),AWS Region,ap-northeast-3,ap-northeast-3,None,None,None,None,None,None,{},UBHUUCQA2BAWFNFA.JRTCKXETXF.6YS6EN2CT7,$0.015 per GB - Asia Pacific (Osaka) Data Tran...,GB,0.015000
1,YD4SUKU86M9QUVVW,None,AmazonS3,Asia Pacific (Sydney),AWS Region,ap-southeast-2,APS2-Tables-SortProcessedBytes,,None,None,None,None,None,None,None,S3-Tables-Sort-ProcessedBytes,Fee for bytes processed for sort or Z-order co...,None,None,None,None,{},YD4SUKU86M9QUVVW.JRTCKXETXF.6YS6EN2CT7,$0.01 per GB fee for bytes processed for sort ...,GB,0.010000
2,E28HMP8J53MB27GF,None,AmazonS3,Asia Pacific (Sydney),AWS Region,ap-southeast-2,APS2-Vectors-Request-Tier3,QueryVectors,None,None,None,None,None,None,None,None,None,S3-API-Vectors-Tier3,Query Vectors API Requests,None,None,{},E28HMP8J53MB27GF.JRTCKXETXF.6YS6EN2CT7,$0.0000027 per Request Query Vectors API Requests,Requests,0.000003
3,7UEPB6UTDGAYV5MV,API Request,AmazonS3,Asia Pacific (Tokyo),AWS Region,ap-northeast-1,APN1-Select-Returned-SIA-Bytes,,None,None,None,None,None,None,None,None,None,S3-API-SIA-Select-Returned,Data Returned by S3 Select in Standard-Infrequ...,None,None,{},7UEPB6UTDGAYV5MV.JRTCKXETXF.6YS6EN2CT7,$0.01 per GB - for bytes returned by S3 Select...,GB,0.010000
4,H5WH6E9K4Y4S7QAU,None,AmazonS3,US West (Oregon),AWS Region,us-west-2,USW2-Vectors-Query-ProcessedBytes-Tier2,QueryVectors,None,None,None,None,None,None,None,S3-Vectors-Query-Processed-Bytes-Tier2,Fee for bytes processed by S3 Vectors Query Ti...,None,None,None,None,{},H5WH6E9K4Y4S7QAU.JRTCKXETXF.6YS6EN2CT7,$0.000001953 per GB fee for bytes processed by...,GB,0.000002


#### Βοηθητικό κώδικας

In [36]:
summary_data = []
for col in df_s3_operations.columns:
    sample_vals = list(df_s3_operations[col].dropna().unique()[:5])
    summary_data.append({'Column': col, 'Sample_Values': sample_vals})

df_summary = pd.DataFrame(summary_data)

# Εμφάνιση χωρίς περικοπές
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
df_summary

,Column,Sample_Values
0,sku,"[UBHUUCQA2BAWFNFA, 5PP56XDP5UX3QP8N, EXBKATS7UXZKUK4R, YD4SUKU86M9QUVVW, E28HMP8J53MB27GF]"
1,attributes.servicecode,[AmazonS3]
2,attributes.transferType,"[IntraRegion Outbound, InterRegion Inbound, InterRegion Outbound, AWS Outbound, AWS Inbound]"
3,attributes.fromLocation,"[Asia Pacific (Osaka), Canada West (Calgary), Asia Pacific (Melbourne), EU (Frankfurt), EU (Ireland)]"
4,attributes.fromLocationType,"[AWS Region, Other, AWS Local Zone]"
5,attributes.toLocation,"[Asia Pacific (Osaka), Asia Pacific (Melbourne), Asia Pacific (Hyderabad), Asia Pacific (Jakarta), Israel (Tel Aviv)]"
6,attributes.toLocationType,"[AWS Region, AWS Edge Location, AWS Local Zone, Other]"
7,attributes.usagetype,"[APN3-APN3-S3RTC-Out-Bytes, APS6-CAN2-S3RTC-In-Bytes, APS5-APS6-S3RTC-In-Bytes, APS2-Tables-SortProcessedBytes, APS2-Vectors-Request-Tier3]"
8,attributes.operation,"[, QueryVectors, UploadPartForRepl, RestoreObject, MRAP-Dtransfer]"
9,attributes.fromRegionCode,"[ap-northeast-3, ca-west-1, ap-southeast-4, eu-central-1, eu-west-1]"
